In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

/tmp/ipykernel_316026/1657699150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = Path("Information_Retrieval_and_Text_Processing_Guide_6308.pdf")
loader = PyMuPDFLoader(str(pdf_path))
docs = loader.load()

In [3]:
!pwd

/home/gauravgupta/codebase/agentic-ai-krish-naik/rag/rag_practice


In [4]:
docs

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-27T04:27:46+00:00', 'source': 'Information_Retrieval_and_Text_Processing_Guide_6308.pdf', 'file_path': 'Information_Retrieval_and_Text_Processing_Guide_6308.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': 'Information Retrieval and Text Processing', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-10-27T04:27:46+00:00', 'trapped': '', 'modDate': "D:20261027042746+00'00'", 'creationDate': "D:20261027042746+00'00'", 'page': 0}, page_content='Information Retrieval & Text\nProcessing\nA Comprehensive Guide to Modern Chunking Strategies\nAbstract\nThis document serves as a standardized testbed for evaluating Natural Language Processing (NLP) text\nchunking strategies. It deliberately incorporates a wide array of typographical and structural elements—\nranging from dense paragraphs and deep hierarchical headings to tabular data and bulleted lists. The\npurpose of this guide is

In [5]:
for doc in docs:
    print(doc.metadata)
    print(doc.page_content)

{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-27T04:27:46+00:00', 'source': 'Information_Retrieval_and_Text_Processing_Guide_6308.pdf', 'file_path': 'Information_Retrieval_and_Text_Processing_Guide_6308.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': 'Information Retrieval and Text Processing', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-10-27T04:27:46+00:00', 'trapped': '', 'modDate': "D:20261027042746+00'00'", 'creationDate': "D:20261027042746+00'00'", 'page': 0}
Information Retrieval & Text
Processing
A Comprehensive Guide to Modern Chunking Strategies
Abstract
This document serves as a standardized testbed for evaluating Natural Language Processing (NLP) text
chunking strategies. It deliberately incorporates a wide array of typographical and structural elements—
ranging from dense paragraphs and deep hierarchical headings to tabular data and bulleted lists. The
purpose of this guide is to provide a robust corpus for algorithm

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter


def fixed_size_chunking(chunk_size, chunk_overlap):
    splitter = CharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
        )
    return splitter

def recursive_character_split_chunking(chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                length_function=len,
                separators=["\n\n", "\n", " ", ""]
            )
    return splitter

def semantic_search_chunking():
    from langchain_experimental.text_splitter import SemanticChunker
    from langchain_community.embeddings import HuggingFaceEmbeddings
    local_emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    semantic_splitter = SemanticChunker(
        local_emb,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )
    return semantic_splitter


In [7]:
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Local embedding model (offline, no API key needed)
# all-MiniLM-L6-v2 -> 384-dim vectors, max 256 tokens per chunk
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dim: {embed_model.get_sentence_embedding_dimension()}")


def embed_chunks(chunks):
    """Turn a list of Document chunks into a 2D float32 numpy array of embeddings.
    We embed the *text* of each chunk; the vector is the numerical meaning of that chunk."""
    texts = [c.page_content for c in chunks]
    vecs = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return np.asarray(vecs, dtype="float32")


def store_chunks(strategy_name, chunks):
    """Embed a set of chunks and save them into a dedicated FAISS index + metadata file.
    Each strategy gets its own folder so we can compare retrieval independently."""
    out_dir = Path(strategy_name)
    out_dir.mkdir(exist_ok=True)

    embeddings = embed_chunks(chunks)
    dim = embeddings.shape[1]
    print(f"[{strategy_name}] embedded {embeddings.shape[0]} chunks -> shape {embeddings.shape}")

    # FAISS index: IndexFlatL2 measures Euclidean distance between query and chunk vectors.
    # Smaller distance = more similar.
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)  # all chunk vectors go into the index

    faiss.write_index(index, str(out_dir / "faiss.index"))

    # We must store the original text alongside each vector, because FAISS only keeps
    # numbers -- it can't tell us *what* a matched chunk says. Metadata bridges that gap.
    metadata = [{"text": c.page_content, "page": c.metadata.get("page")} for c in chunks]
    with open(out_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)

    print(f"[{strategy_name}] saved FAISS index + metadata to ./{out_dir}/")
    return out_dir


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2527.65it/s]


Embedding dim: 384


/tmp/ipykernel_316026/4054006072.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {embed_model.get_sentence_embedding_dimension()}")


In [8]:
# ---- 1) Fixed-size chunking ----
fixed_splitter = fixed_size_chunking(chunk_size=1000, chunk_overlap=200)
fixed_chunks = fixed_splitter.split_documents(docs)
print(f"Fixed-size chunks: {len(fixed_chunks)}")

# ---- 2) Recursive chunking ----
recursive_splitter = recursive_character_split_chunking(chunk_size=1000, chunk_overlap=200)
recursive_chunks = recursive_splitter.split_documents(docs)
print(f"Recursive chunks: {len(recursive_chunks)}")

# ---- 3) Semantic chunking ----
# NOTE: SemanticChunker here is configured with OpenAIEmbeddings in your chunking.py.
# Since we are going fully local, we instead build a SemanticChunker with the local model.
semantic_splitter = semantic_search_chunking()
semantic_chunks = semantic_splitter.split_documents(docs)
print(f"Semantic chunks: {len(semantic_chunks)}")


/tmp/ipykernel_316026/3667582470.py:21: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
/tmp/ipykernel_316026/3667582470.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  local_emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Fixed-size chunks: 15
Recursive chunks: 28


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2113.23it/s]


Semantic chunks: 30


In [9]:
# Store each strategy's chunks in its own FAISS index folder.
store_chunks("faiss_fixed", fixed_chunks)
store_chunks("faiss_recursive", recursive_chunks)
store_chunks("faiss_semantic", semantic_chunks)


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]


[faiss_fixed] embedded 15 chunks -> shape (15, 384)
[faiss_fixed] saved FAISS index + metadata to ./faiss_fixed/


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


[faiss_recursive] embedded 28 chunks -> shape (28, 384)
[faiss_recursive] saved FAISS index + metadata to ./faiss_recursive/


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]

[faiss_semantic] embedded 30 chunks -> shape (30, 384)
[faiss_semantic] saved FAISS index + metadata to ./faiss_semantic/


PosixPath('faiss_semantic')

In [13]:
# 10 test questions.
# Each maps to ONE specific page of the PDF (the 'source_page' is the ground-truth
# page where the answer lives). This lets us check, per strategy, whether the most
# relevant chunk is returned at rank 1.
test_questions = [
    {
        "question": "What is Information Retrieval (IR) and why is chunking needed for long documents?",
        "source_page": 3,   # 1. Introduction to Information Retrieval
    },
    {
        "question": "How is Okapi BM25 different from TF-IDF, and what limitation does it still have?",
        "source_page": 4,   # 2. Evolution of Search Systems
    },
    {
        "question": "What four structural elements should a document parser identify before chunking?",
        "source_page": 5,   # 3. Fundamentals of Document Parsing
    },
    {
        "question": "In the organizational hierarchy example, which two business units report to the CFO?",
        "source_page": 6,   # 4. Structural Elements in Text
    },
    {
        "question": "What are the pros and cons of the recursive character chunking strategy?",
        "source_page": 7,   # 5. Comparison of Common Strategies
    },
    {
        "question": "What is the 'hard cut problem' in fixed-size chunking?",
        "source_page": 8,   # 6. Fixed-Size Chunking Algorithms
    },
    {
        "question": "With a chunk size of 500 tokens and overlap of 100, what token range does chunk 2 cover?",
        "source_page": 9,   # 7. Overlap and Sliding Windows
    },
    {
        "question": "What was the net profit in Q3 2023 according to the sample financial data table?",
        "source_page": 11,  # 9. Handling Tabular Data
    },
    {
        "question": "What is the first step in the server shutdown procedure for the database?",
        "source_page": 12,  # 10. Processing Hierarchical Lists
    },
    {
        "question": "What is the 'dilution effect' that occurs when embedding a very large chunk?",
        "source_page": 13,  # 11. Vector Embeddings and Chunk Size
    },
]

print(f"Defined {len(test_questions)} test questions.")
for q in test_questions:
    print(f"  p{q['source_page']:>2}: {q['question']}")


Defined 10 test questions.
  p 3: What is Information Retrieval (IR) and why is chunking needed for long documents?
  p 4: How is Okapi BM25 different from TF-IDF, and what limitation does it still have?
  p 5: What four structural elements should a document parser identify before chunking?
  p 6: In the organizational hierarchy example, which two business units report to the CFO?
  p 7: What are the pros and cons of the recursive character chunking strategy?
  p 8: What is the 'hard cut problem' in fixed-size chunking?
  p 9: With a chunk size of 500 tokens and overlap of 100, what token range does chunk 2 cover?
  p11: What was the net profit in Q3 2023 according to the sample financial data table?
  p12: What is the first step in the server shutdown procedure for the database?
  p13: What is the 'dilution effect' that occurs when embedding a very large chunk?


In [14]:
import os
import faiss
import pickle
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# ---- Connect to the LLM (same pattern as rag/src/search.py) ----
llm = ChatOpenAI(
    model="minimax-local",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("CHAT_API_BASE_URL"),
    temperature=0,
)
print("LLM connected.")


# ---- Load a stored FAISS index + its metadata ----
def load_store(store_dir):
    d = Path(store_dir)
    index = faiss.read_index(str(d / "faiss.index"))
    metadata = pickle.load(open(d / "metadata.pkl", "rb"))
    return index, metadata


# ---- Retrieve top-k chunks for a query from one store ----
def retrieve(store_dir, query, top_k=1):
    index, metadata = load_store(store_dir)
    q_vec = np.asarray(embed_model.encode([query], convert_to_numpy=True), dtype="float32")
    # D = distances (lower = closer), I = indices into the index/metadata lists
    D, I = index.search(q_vec, top_k)
    hits = []
    for dist, idx in zip(D[0], I[0]):
        hits.append({**metadata[idx], "distance": float(dist), "rank": len(hits) + 1})
    return hits


# ---- LLM-as-judge: does the retrieved chunk answer the question? ----
def llm_judges_relevant(question, chunk_text):
    prompt = (
        f"You are a strict retrieval judge. Decide whether the CHUNK below contains "
        f"enough information to fully answer the QUESTION. Reply with only YES or NO.\n\n"
        f"QUESTION: {question}\n\n"
        f"CHUNK: {chunk_text[:1500]}\n\n"
        f"VERDICT:"
    )
    out = llm.invoke([prompt]).content.strip().upper()
    return out.startswith("YES")


# ---- Run the experiment across all 3 strategies ----
strategies = ["faiss_fixed", "faiss_recursive", "faiss_semantic"]

results = {s: [] for s in strategies}   # per-strategy list of (question_idx, relevant)
for qi, q in enumerate(test_questions):
    for s in strategies:
        top = retrieve(s, q["question"], top_k=1)[0]
        relevant = llm_judges_relevant(q["question"], top["text"])
        results[s].append(relevant)
        print(f"Q{qi+1:>2} [{s:<14}] page={top.get('page'):>2}  relevant={relevant}")

print()
print("=" * 60)
print("Strategy  | rank-1 relevant chunks (out of 10)")
print("=" * 60)
for s in strategies:
    score = sum(results[s])
    print(f"{s:<14} | {score}/10")
print("=" * 60)

best = max(strategies, key=lambda s: sum(results[s]))
print(f"\nBest chunking strategy for rank-1 retrieval: {best}")


LLM connected.
Q 1 [faiss_fixed   ] page= 2  relevant=True
Q 1 [faiss_recursive] page= 3  relevant=False
Q 1 [faiss_semantic] page= 3  relevant=False
Q 2 [faiss_fixed   ] page= 3  relevant=True
Q 2 [faiss_recursive] page= 3  relevant=False
Q 2 [faiss_semantic] page= 3  relevant=True
Q 3 [faiss_fixed   ] page= 4  relevant=True
Q 3 [faiss_recursive] page= 4  relevant=True
Q 3 [faiss_semantic] page= 4  relevant=False
Q 4 [faiss_fixed   ] page= 5  relevant=True
Q 4 [faiss_recursive] page= 5  relevant=False
Q 4 [faiss_semantic] page= 5  relevant=True
Q 5 [faiss_fixed   ] page= 7  relevant=False
Q 5 [faiss_recursive] page= 7  relevant=False
Q 5 [faiss_semantic] page= 7  relevant=False
Q 6 [faiss_fixed   ] page= 7  relevant=True
Q 6 [faiss_recursive] page= 8  relevant=True
Q 6 [faiss_semantic] page= 7  relevant=True
Q 7 [faiss_fixed   ] page= 8  relevant=True
Q 7 [faiss_recursive] page= 8  relevant=True
Q 7 [faiss_semantic] page= 8  relevant=True
Q 8 [faiss_fixed   ] page=10  relevant=True
Q 

In [15]:
# Cross-page test set: 10 questions whose answers require semantic info SPREAD
# across multiple pages. Unlike the single-page set, no one chunk fully answers
# these -- the correct context is distributed, so we judge whether the TOP-K
# retrieved chunks TOGETHER cover the answer. This stresses how well each strategy
# keeps semantically-related material retrievable.
# source_pages: 0-indexed page numbers (matching metadata['page']) whose combined
# content forms the answer.
test_questions_cross = [
    {
        "question": "How do embedding-model context-window limits drive the need for chunking, and how should those same limits influence the chunk size you choose?",
        "source_pages": [2, 3, 12],   # intro/chunking need, dense-model token limits, chunk size & model limits
    },
    {
        "question": "What is the hard-cut problem in fixed-size chunking, and how does overlap/sliding-window help fix it and at what storage or cost?",
        "source_pages": [7, 8],      # fixed-size hard cut, overlap mechanics + storage/cost
    },
    {
        "question": "How is semantic boundary detection performed, and how does it relate to keeping related list items together in one chunk?",
        "source_pages": [9, 11],     # semantic boundary detection, hierarchical lists
    },
    {
        "question": "Why does poor chunking lead to poor retrieval even with advanced models, and what structural elements should a parser preserve to prevent this?",
        "source_pages": [2, 4],      # key takeaway (poor chunking->poor retrieval), structural parsing steps
    },
    {
        "question": "How do dense vector embeddings make chunk boundaries critical for retrieval, and what is the dilution effect of embedding very large chunks?",
        "source_pages": [3, 12],     # dense vector search, dilution effect
    },
    {
        "question": "Compare fixed-size, recursive, and semantic chunking (pros and cons), and explain how the downstream application should drive the choice of strategy.",
        "source_pages": [6, 14],     # strategy comparison table, conclusion/downstream
    },
    {
        "question": "Why does token-based chunking align better with model context windows than character-based chunking, and what token limits of the model matter?",
        "source_pages": [7, 12],     # char vs token measurement, model context limits
    },
    {
        "question": "How do naive chunkers destroy structured content such as tables and hierarchical lists, and what should a structure-aware chunker do instead?",
        "source_pages": [5, 10, 11], # structural elements, tabular data, hierarchical lists
    },
    {
        "question": "How can OCR errors and upstream parsing quality propagate downstream to hurt chunking and embedding?",
        "source_pages": [4, 14],     # OCR/parsing, chunking as foundational bottleneck
    },
    {
        "question": "How do retrieval metrics and generation metrics together tell you whether a chunking strategy is effective?",
        "source_pages": [13, 14],    # retrieval + generation metrics, chunking as bottleneck
    },
]

print(f"Defined {len(test_questions_cross)} cross-page test questions.")
for q in test_questions_cross:
    print(f"  pages {q['source_pages']}: {q['question'][:70]}...")


Defined 10 cross-page test questions.
  pages [2, 3, 12]: How do embedding-model context-window limits drive the need for chunki...
  pages [7, 8]: What is the hard-cut problem in fixed-size chunking, and how does over...
  pages [9, 11]: How is semantic boundary detection performed, and how does it relate t...
  pages [2, 4]: Why does poor chunking lead to poor retrieval even with advanced model...
  pages [3, 12]: How do dense vector embeddings make chunk boundaries critical for retr...
  pages [6, 14]: Compare fixed-size, recursive, and semantic chunking (pros and cons), ...
  pages [7, 12]: Why does token-based chunking align better with model context windows ...
  pages [5, 10, 11]: How do naive chunkers destroy structured content such as tables and hi...
  pages [4, 14]: How can OCR errors and upstream parsing quality propagate downstream t...
  pages [13, 14]: How do retrieval metrics and generation metrics together tell you whet...


In [16]:
# ---- Evaluate the CROSS-PAGE test set ----
# For each question we retrieve TOP_K chunks (not just 1), because the answer is
# spread across pages. The LLM judges whether the retrieved context TOGETHER
# contains enough info to answer.
TOP_K = 3

def llm_judges_context_sufficient(question, context):
    prompt = (
        "You are a strict retrieval judge. The CONTEXT below is a set of chunks "
        "retrieved from a document. Decide whether the CONTEXT contains enough "
        "information to fully answer the QUESTION. Reply with only YES or NO.\n\n"
        f"QUESTION: {question}\n\n"
        f"CONTEXT:\n{context[:4000]}\n\n"
        f"VERDICT:"
    )
    out = llm.invoke([prompt]).content.strip().upper()
    return out.startswith("YES")


cross_results = {s: [] for s in strategies}
for qi, q in enumerate(test_questions_cross):
    for s in strategies:
        top = retrieve(s, q["question"], top_k=TOP_K)
        context = "\n\n---\n\n".join(hit["text"] for hit in top)
        retrieved_pages = sorted({hit.get("page") for hit in top})
        sufficient = llm_judges_context_sufficient(q["question"], context)
        cross_results[s].append(sufficient)
        print(f"Q{qi+1:>2} [{s:<14}] pages={retrieved_pages}  sufficient={sufficient}")

print()
print("=" * 60)
print(f"Cross-page set | rank-{TOP_K} context sufficient (out of 10)")
print("=" * 60)
for s in strategies:
    score = sum(cross_results[s])
    print(f"{s:<14} | {score}/10")
print("=" * 60)

best_cross = max(strategies, key=lambda s: sum(cross_results[s]))
print(f"\nBest strategy when the answer is spread across pages: {best_cross}")


Q 1 [faiss_fixed   ] pages=[7, 8, 12]  sufficient=True
Q 1 [faiss_recursive] pages=[2, 12]  sufficient=True
Q 1 [faiss_semantic] pages=[7, 12]  sufficient=True
Q 2 [faiss_fixed   ] pages=[7, 8, 14]  sufficient=True
Q 2 [faiss_recursive] pages=[8, 12, 14]  sufficient=True
Q 2 [faiss_semantic] pages=[7, 8, 14]  sufficient=True
Q 3 [faiss_fixed   ] pages=[1, 8, 9]  sufficient=True
Q 3 [faiss_recursive] pages=[4, 9, 11]  sufficient=True
Q 3 [faiss_semantic] pages=[1, 9]  sufficient=True
Q 4 [faiss_fixed   ] pages=[0, 5, 14]  sufficient=True
Q 4 [faiss_recursive] pages=[2, 5, 14]  sufficient=True
Q 4 [faiss_semantic] pages=[3, 12, 14]  sufficient=True
Q 5 [faiss_fixed   ] pages=[7, 8, 12]  sufficient=True
Q 5 [faiss_recursive] pages=[8, 12]  sufficient=True
Q 5 [faiss_semantic] pages=[8, 12]  sufficient=True
Q 6 [faiss_fixed   ] pages=[5, 6, 14]  sufficient=True
Q 6 [faiss_recursive] pages=[5, 6, 14]  sufficient=False
Q 6 [faiss_semantic] pages=[5, 6, 14]  sufficient=False
Q 7 [faiss_fixed 

In [17]:
# Focused-concept test set: 10 questions that each target ONE tightly-focused,
# semantically-aligned concept. These are deliberately phrased with vocabulary that
# matches the *isolated* semantic sub-chunk (SemanticChunker splits each page into
# topically-pure sub-chunks). For fixed-size (1 chunk/page) and recursive (larger
# paragraph chunks), the same fact is buried inside a bigger blended chunk, which
# dilutes its embedding -- so these should favor the semantic strategy.
test_questions_focused = [
    {
        "question": "When a chunk is very large, its embedding becomes a 'blended smoothie' of many different concepts. What is this dilution effect and why does it lower similarity to a specific query?",
        "source_page": 12,
    },
    {
        "question": "At retrieval time the system retrieves the most relevant chunks rather than the entire document. Why does this make the boundary of the chunk a critical factor in the system's success?",
        "source_page": 3,
    },
    {
        "question": "Why does a legal document analysis tool require document-based chunking to preserve clause boundaries, while a general knowledge chatbot might perform adequately with recursive character chunking?",
        "source_page": 6,
    },
    {
        "question": "Why can overlap lead to duplicate information being fed into the generative LLM when multiple overlapping chunks are retrieved for the same query?",
        "source_page": 8,
    },
    {
        "question": "What does the Context Precision metric measure, and what does finding a lot of irrelevant noise in retrieved chunks often indicate about chunk size?",
        "source_page": 13,
    },
    {
        "question": "Why does flattening a table into plain text and then splitting it arbitrarily permanently destroy the relationship between a column header and its corresponding data cell?",
        "source_page": 10,
    },
    {
        "question": "Why should bulleted and numbered lists, which contain highly structured information, ideally be kept together or chunked logically instead of being split?",
        "source_page": 4,
    },
    {
        "question": "Why does the document itself serve as a test case for the hard-cut problem, where a naive algorithm set to a low character limit slices through paragraphs awkwardly?",
        "source_page": 7,
    },
    {
        "question": "Why is the ideal chunk size not determined in a vacuum, but heavily influenced by the specific embedding model being used in the RAG pipeline?",
        "source_page": 12,
    },
    {
        "question": "Why can running your chunking algorithms against this document let you observe firsthand how your pipeline handles headers, lists, tabular data, and varying paragraph lengths?",
        "source_page": 14,
    },
]

print(f"Defined {len(test_questions_focused)} focused-concept questions.")
for q in test_questions_focused:
    print(f"  p{q['source_page']:>2}: {q['question'][:75]}...")


Defined 10 focused-concept questions.
  p12: When a chunk is very large, its embedding becomes a 'blended smoothie' of m...
  p 3: At retrieval time the system retrieves the most relevant chunks rather than...
  p 6: Why does a legal document analysis tool require document-based chunking to ...
  p 8: Why can overlap lead to duplicate information being fed into the generative...
  p13: What does the Context Precision metric measure, and what does finding a lot...
  p10: Why does flattening a table into plain text and then splitting it arbitrari...
  p 4: Why should bulleted and numbered lists, which contain highly structured inf...
  p 7: Why does the document itself serve as a test case for the hard-cut problem,...
  p12: Why is the ideal chunk size not determined in a vacuum, but heavily influen...
  p14: Why can running your chunking algorithms against this document let you obse...


In [18]:
# ---- Evaluate the FOCUSED-CONCEPT test set ----
# These questions are designed so the answer is a single tightly-focused concept that
# SemanticChunker isolates into its own topically-pure sub-chunk. We judge by rank-1
# retrieval + LLM verdict (same as the single-page set), expecting semantic to win.
focused_results = {s: [] for s in strategies}
for qi, q in enumerate(test_questions_focused):
    for s in strategies:
        top = retrieve(s, q["question"], top_k=1)[0]
        relevant = llm_judges_relevant(q["question"], top["text"])
        focused_results[s].append(relevant)
        print(f"Q{qi+1:>2} [{s:<14}] page={top.get('page'):>2}  relevant={relevant}")

print()
print("=" * 60)
print("Focused-concept set | rank-1 relevant (out of 10)")
print("=" * 60)
for s in strategies:
    score = sum(focused_results[s])
    print(f"{s:<14} | {score}/10")
print("=" * 60)

best_focused = max(strategies, key=lambda s: sum(focused_results[s]))
print(f"\nBest strategy for tightly-focused semantic questions: {best_focused}")

print("\n===== SUMMARY ACROSS ALL THREE TEST SETS =====")
for s in strategies:
    print(f"{s:<14} | single-page {sum(results[s])}/10 | "
          f"cross-page {sum(cross_results[s])}/10 | "
          f"focused {sum(focused_results[s])}/10")


Q 1 [faiss_fixed   ] page=12  relevant=True
Q 1 [faiss_recursive] page=12  relevant=False
Q 1 [faiss_semantic] page=12  relevant=True
Q 2 [faiss_fixed   ] page=13  relevant=False
Q 2 [faiss_recursive] page= 3  relevant=True
Q 2 [faiss_semantic] page= 3  relevant=False
Q 3 [faiss_fixed   ] page= 6  relevant=True
Q 3 [faiss_recursive] page= 6  relevant=True
Q 3 [faiss_semantic] page= 6  relevant=True
Q 4 [faiss_fixed   ] page= 8  relevant=True
Q 4 [faiss_recursive] page= 8  relevant=True
Q 4 [faiss_semantic] page= 8  relevant=True
Q 5 [faiss_fixed   ] page=13  relevant=True
Q 5 [faiss_recursive] page=13  relevant=True
Q 5 [faiss_semantic] page=13  relevant=True
Q 6 [faiss_fixed   ] page=10  relevant=True
Q 6 [faiss_recursive] page=10  relevant=True
Q 6 [faiss_semantic] page=10  relevant=True
Q 7 [faiss_fixed   ] page=11  relevant=True
Q 7 [faiss_recursive] page= 4  relevant=True
Q 7 [faiss_semantic] page= 4  relevant=True
Q 8 [faiss_fixed   ] page= 9  relevant=False
Q 8 [faiss_recursive]